# AMRO Fit Results Analysis

This notebook analyzes the fitted parameters from AMRO (Angle-resolved Magnetoresistance Oscillation) measurements.

## Background

The AMRO data was fitted using:

$$\rho_{xx}(\theta) = \rho_0\left(1 + \sum_i a_i\sin(f_i\theta+\phi_i)\right)$$

where:
- $\rho_0$ is the mean resistivity
- $a_i$ is the amplitude of the $i$-th symmetry component
- $f_i$ is the frequency (2 for 2-fold, 4 for 4-fold symmetry, etc.)
- $\phi_i$ is the phase

## Analysis Goals

1. **Temperature dependence** - How do amplitudes evolve with T at fixed H?
2. **Field dependence** - How do amplitudes evolve with H at fixed T?
3. **Amplitude ratios** - Track $a_4/a_2$ to identify Fermi surface changes
4. **Phase analysis** - Look for systematic phase shifts
5. **Geometry comparison** - Compare parallel vs perpendicular orientations

# 1. Setup and Data Loading

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from amro import AMROLoader

# Plot settings
sns.set_context('notebook')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Load the project data (assumes notebook 01 has been run and data is pickled)
loader = AMROLoader('AMRO_combined_data')
project_data = loader.load_amro()

print(project_data.get_summary_statistics())

In [ ]:
# Extract fit results as DataFrame for analysis
# Note: This requires fits to have been completed in notebook 01
fit_df = project_data.get_fit_results_as_df()
print(f"Total fit results: {len(fit_df)}")
fit_df.head()

In [ ]:
# Inspect available columns
print("Available columns:")
for col in fit_df.columns:
    print(f"  - {col}")

In [ ]:
# Overview of experimental conditions
print("Experiments:", fit_df['exp_label'].unique())
print("Temperatures (K):", sorted(fit_df['T (K)'].unique()))
print("Magnetic fields (T):", sorted(fit_df['H (T)'].unique()))
print("Geometries:", fit_df['geometry'].unique())

# 2. Temperature Dependence Analysis

Examine how the fitted amplitudes and mean resistivity change with temperature at fixed magnetic field values.

In [ ]:
# Helper function to extract amplitude columns
def get_amplitude_columns(df):
    """Find all amplitude columns in the dataframe."""
    return [col for col in df.columns if col.startswith('a_')]

In [ ]:
# Mean resistivity vs temperature
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for exp_label in fit_df['exp_label'].unique():
    exp_df = fit_df[fit_df['exp_label'] == exp_label]
    
    for h_val in sorted(exp_df['H (T)'].unique()):
        subset = exp_df[exp_df['H (T)'] == h_val].sort_values('T (K)')
        ax_idx = 0 if 'para' in exp_df['geometry'].values[0] else 1
        axes[ax_idx].plot(subset['T (K)'], subset['mean'], 
                         marker='o', label=f'H={h_val}T')

axes[0].set_title('Parallel Geometry')
axes[1].set_title('Perpendicular Geometry')

for ax in axes:
    ax.set_xlabel('Temperature (K)')
    ax.set_ylabel(r'Mean Resistivity $\rho_0$ (ohm-cm)')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Amplitude vs temperature for each symmetry component
amp_cols = get_amplitude_columns(fit_df)

if amp_cols:
    n_amps = len(amp_cols)
    fig, axes = plt.subplots(1, n_amps, figsize=(5*n_amps, 5))
    if n_amps == 1:
        axes = [axes]
    
    for ax, amp_col in zip(axes, amp_cols):
        for exp_label in fit_df['exp_label'].unique():
            exp_df = fit_df[fit_df['exp_label'] == exp_label]
            
            for h_val in sorted(exp_df['H (T)'].unique()):
                subset = exp_df[exp_df['H (T)'] == h_val].sort_values('T (K)')
                if amp_col in subset.columns:
                    ax.plot(subset['T (K)'], subset[amp_col], 
                           marker='o', label=f'{exp_label}, H={h_val}T')
        
        ax.set_xlabel('Temperature (K)')
        ax.set_ylabel(f'Amplitude {amp_col}')
        ax.set_title(f'{amp_col} vs Temperature')
        ax.legend(fontsize='small')
    
    plt.tight_layout()
    plt.show()
else:
    print("No amplitude columns found. Check fit results.")

# 3. Magnetic Field Dependence Analysis

Examine how the fitted amplitudes change with magnetic field at fixed temperatures.

In [ ]:
# Amplitude vs magnetic field for each temperature
amp_cols = get_amplitude_columns(fit_df)

if amp_cols:
    n_amps = len(amp_cols)
    fig, axes = plt.subplots(1, n_amps, figsize=(5*n_amps, 5))
    if n_amps == 1:
        axes = [axes]
    
    for ax, amp_col in zip(axes, amp_cols):
        for exp_label in fit_df['exp_label'].unique():
            exp_df = fit_df[fit_df['exp_label'] == exp_label]
            
            for t_val in sorted(exp_df['T (K)'].unique()):
                subset = exp_df[exp_df['T (K)'] == t_val].sort_values('H (T)')
                if amp_col in subset.columns:
                    ax.plot(subset['H (T)'], subset[amp_col], 
                           marker='s', label=f'{exp_label}, T={t_val}K')
        
        ax.set_xlabel('Magnetic Field (T)')
        ax.set_ylabel(f'Amplitude {amp_col}')
        ax.set_title(f'{amp_col} vs Magnetic Field')
        ax.legend(fontsize='small')
    
    plt.tight_layout()
    plt.show()
else:
    print("No amplitude columns found.")

# 4. Amplitude Ratio Analysis

The ratio of amplitudes (e.g., $a_4/a_2$) is particularly valuable because:
- It is independent of sample geometry uncertainties
- Changes indicate Fermi surface topology changes
- Can reveal Lifshitz transitions or other phase boundaries

In [ ]:
# Calculate amplitude ratios
# Adjust column names based on your actual fit results

amp_cols = get_amplitude_columns(fit_df)
print(f"Available amplitude columns: {amp_cols}")

# Example: if you have a_2 and a_4 columns
if 'a_4' in fit_df.columns and 'a_2' in fit_df.columns:
    fit_df['a4_a2_ratio'] = fit_df['a_4'] / fit_df['a_2']
    print("Created a4_a2_ratio column")
else:
    print("Amplitude columns not found. Modify column names as needed.")
    print("You may need to pivot the data or extract amplitudes differently.")

In [ ]:
# Amplitude ratio vs temperature
if 'a4_a2_ratio' in fit_df.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    for exp_label in fit_df['exp_label'].unique():
        exp_df = fit_df[fit_df['exp_label'] == exp_label]
        
        for h_val in sorted(exp_df['H (T)'].unique()):
            subset = exp_df[exp_df['H (T)'] == h_val].sort_values('T (K)')
            ax.plot(subset['T (K)'], subset['a4_a2_ratio'], 
                   marker='o', label=f'{exp_label}, H={h_val}T')
    
    ax.set_xlabel('Temperature (K)')
    ax.set_ylabel(r'$a_4/a_2$ Ratio')
    ax.set_title('Amplitude Ratio vs Temperature')
    ax.legend()
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Heatmap: T vs H colored by amplitude ratio
# This visualization helps identify phase boundaries

if 'a4_a2_ratio' in fit_df.columns:
    for exp_label in fit_df['exp_label'].unique():
        exp_df = fit_df[fit_df['exp_label'] == exp_label]
        
        # Pivot for heatmap
        pivot_df = exp_df.pivot_table(
            index='T (K)', 
            columns='H (T)', 
            values='a4_a2_ratio',
            aggfunc='mean'
        )
        
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(pivot_df, annot=True, fmt='.3f', cmap='RdBu_r', 
                   center=0, ax=ax, cbar_kws={'label': r'$a_4/a_2$'})
        ax.set_title(f'{exp_label}: Amplitude Ratio Phase Diagram')
        ax.set_xlabel('Magnetic Field (T)')
        ax.set_ylabel('Temperature (K)')
        
        plt.tight_layout()
        plt.show()

# 5. Phase Analysis

The phases $\phi_i$ can provide information about:
- Rotation of the Fermi surface in k-space
- Changes in the dominant conduction channel
- Sample alignment verification

In [ ]:
# Find phase columns
phase_cols = [col for col in fit_df.columns if col.startswith('phi_') or col.startswith('phase_')]
print(f"Phase columns found: {phase_cols}")

In [ ]:
# Phase vs temperature
if phase_cols:
    n_phases = len(phase_cols)
    fig, axes = plt.subplots(1, n_phases, figsize=(5*n_phases, 5))
    if n_phases == 1:
        axes = [axes]
    
    for ax, phase_col in zip(axes, phase_cols):
        for exp_label in fit_df['exp_label'].unique():
            exp_df = fit_df[fit_df['exp_label'] == exp_label]
            
            for h_val in sorted(exp_df['H (T)'].unique()):
                subset = exp_df[exp_df['H (T)'] == h_val].sort_values('T (K)')
                ax.plot(subset['T (K)'], np.degrees(subset[phase_col]), 
                       marker='o', label=f'{exp_label}, H={h_val}T')
        
        ax.set_xlabel('Temperature (K)')
        ax.set_ylabel(f'{phase_col} (degrees)')
        ax.set_title(f'{phase_col} vs Temperature')
        ax.legend(fontsize='small')
    
    plt.tight_layout()
    plt.show()
else:
    print("No phase columns found in fit results.")

# 6. Geometry Comparison

Compare results between parallel and perpendicular measurement geometries.

- **Parallel (para)**: Current vector becomes parallel to B at 90 degrees
- **Perpendicular (perp)**: Current vector remains orthogonal to B throughout rotation

In [ ]:
# Side-by-side comparison of geometries
geometries = fit_df['geometry'].unique()

if len(geometries) >= 2:
    amp_cols = get_amplitude_columns(fit_df)
    
    if amp_cols:
        # Use the first amplitude column for comparison
        amp_col = amp_cols[0]
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Temperature dependence comparison
        ax = axes[0]
        for geo in geometries:
            geo_df = fit_df[fit_df['geometry'] == geo]
            # Average over all H values
            avg_df = geo_df.groupby('T (K)')[amp_col].mean().reset_index()
            ax.plot(avg_df['T (K)'], avg_df[amp_col], 
                   marker='o', label=f'{geo}')
        
        ax.set_xlabel('Temperature (K)')
        ax.set_ylabel(f'{amp_col} (H-averaged)')
        ax.set_title('Geometry Comparison: T-dependence')
        ax.legend()
        
        # Field dependence comparison
        ax = axes[1]
        for geo in geometries:
            geo_df = fit_df[fit_df['geometry'] == geo]
            # Average over all T values
            avg_df = geo_df.groupby('H (T)')[amp_col].mean().reset_index()
            ax.plot(avg_df['H (T)'], avg_df[amp_col], 
                   marker='s', label=f'{geo}')
        
        ax.set_xlabel('Magnetic Field (T)')
        ax.set_ylabel(f'{amp_col} (T-averaged)')
        ax.set_title('Geometry Comparison: H-dependence')
        ax.legend()
        
        plt.tight_layout()
        plt.show()
else:
    print(f"Only one geometry found: {geometries}")

# 7. Fit Quality Assessment

Examine the quality of fits through chi-squared values and parameter uncertainties.

In [ ]:
# Chi-squared distribution
if 'chi_sq' in fit_df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    ax = axes[0]
    ax.hist(fit_df['chi_sq'], bins=20, edgecolor='black', alpha=0.7)
    ax.set_xlabel(r'$\chi^2$')
    ax.set_ylabel('Count')
    ax.set_title('Distribution of Fit Quality')
    ax.axvline(fit_df['chi_sq'].median(), color='red', 
               linestyle='--', label=f"Median: {fit_df['chi_sq'].median():.3f}")
    ax.legend()
    
    # Chi-squared vs conditions
    ax = axes[1]
    for exp_label in fit_df['exp_label'].unique():
        exp_df = fit_df[fit_df['exp_label'] == exp_label]
        ax.scatter(exp_df['T (K)'], exp_df['chi_sq'], 
                  label=exp_label, alpha=0.7, s=exp_df['H (T)']*20)
    
    ax.set_xlabel('Temperature (K)')
    ax.set_ylabel(r'$\chi^2$')
    ax.set_title(r'Fit Quality vs Temperature (size $\propto$ H)')
    ax.legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("Chi-squared column not found.")

In [ ]:
# Identify poor fits for potential re-examination
if 'chi_sq' in fit_df.columns:
    threshold = fit_df['chi_sq'].quantile(0.9)  # Top 10% worst fits
    poor_fits = fit_df[fit_df['chi_sq'] > threshold][['exp_label', 'T (K)', 'H (T)', 'chi_sq']]
    
    print(f"Fits with chi_sq > {threshold:.4f} (top 10%):")
    print(poor_fits.to_string(index=False))

# 8. Summary and Conclusions

Document key observations from the analysis.

In [ ]:
# Summary statistics
print("=" * 60)
print("FIT RESULTS SUMMARY")
print("=" * 60)
print(f"Total oscillations fitted: {len(fit_df)}")
print(f"Experiments: {list(fit_df['exp_label'].unique())}")
print(f"Temperature range: {fit_df['T (K)'].min()} - {fit_df['T (K)'].max()} K")
print(f"Field range: {fit_df['H (T)'].min()} - {fit_df['H (T)'].max()} T")

if 'chi_sq' in fit_df.columns:
    print(f"\nFit quality (chi_sq):")
    print(f"  Mean: {fit_df['chi_sq'].mean():.4f}")
    print(f"  Median: {fit_df['chi_sq'].median():.4f}")
    print(f"  Std: {fit_df['chi_sq'].std():.4f}")

print("=" * 60)

## Key Observations

*Fill in after running the analysis:*

1. **Temperature dependence:**
   - 

2. **Field dependence:**
   - 

3. **Amplitude ratios:**
   - 

4. **Phase behavior:**
   - 

5. **Geometry comparison:**
   - 

# 9. Export Results (Optional)

In [ ]:
# Export processed data for external use
# Uncomment to save

# fit_df.to_csv('../data/final/fit_results_analysis.csv', index=False)
# print("Exported fit results to CSV")